# Car Paint Defect Detection — YOLOv8 训练

**使用说明（上传到 Kaggle 前先做）：**
1. Kaggle → Datasets → New Dataset，上传整个 `final year car paint defect.v1i.yolov8/` 文件夹
2. 新建 Notebook，右侧 Add Data 添加刚才的数据集
3. 右侧 Session options → Accelerator 选 **GPU P100**
4. 粘贴本 Notebook 内容，依次运行

## 1. 确认环境与数据集路径

In [ ]:
import os
import glob

# 查看 Kaggle 挂载的数据集
!ls /kaggle/input/

# 找到数据集根目录（会自动识别）
candidates = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
print('找到 data.yaml:', candidates)
DATASET_ROOT = os.path.dirname(candidates[0])
print('数据集根目录:', DATASET_ROOT)

## 2. 安装 Ultralytics

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

## 3. 修正 data.yaml 路径

原始 data.yaml 使用相对路径，Kaggle 环境需要改为绝对路径。

In [ ]:
import yaml
import shutil

# 读取原始 yaml
with open(os.path.join(DATASET_ROOT, 'data.yaml'), 'r') as f:
    data = yaml.safe_load(f)

print('原始内容:', data)

# 改为绝对路径，复制到可写目录
data['train'] = os.path.join(DATASET_ROOT, 'train', 'images')
data['val']   = os.path.join(DATASET_ROOT, 'valid', 'images')
data['test']  = os.path.join(DATASET_ROOT, 'test',  'images')

YAML_PATH = '/kaggle/working/data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data, f, allow_unicode=True)

print('\n修正后内容:')
with open(YAML_PATH) as f:
    print(f.read())

## 4. 开始训练

| 模型 | 参数量 | 速度 | 精度 | 推荐场景 |
|------|--------|------|------|----------|
| yolov8n | 3.2M | 最快 | 一般 | 快速验证 |
| yolov8s | 11.2M | 快 | 较好 | **毕设推荐** |
| yolov8m | 25.9M | 中等 | 好 | 精度优先 |

In [ ]:
from ultralytics import YOLO

# 加载预训练权重（自动从官网下载，约 22MB）
model = YOLO('yolov8s.pt')

# 开始训练
results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,           # GPU
    project='/kaggle/working/runs',
    name='car_defect',
    patience=20,        # 20轮无提升则早停
    save=True,
    plots=True,         # 生成训练曲线图
    val=True,
)

print('训练完成！')
print('最优权重路径:', results.save_dir)

ModuleNotFoundError: No module named 'ultralytics'

## 5. 在测试集上评估

In [ ]:
import os

best_weights = os.path.join(results.save_dir, 'weights', 'best.pt')
model_best = YOLO(best_weights)

# 在测试集上计算 mAP
metrics = model_best.val(
    data=YAML_PATH,
    split='test',
    device=0,
)

print(f"\n测试集结果：")
print(f"  mAP@50    : {metrics.box.map50:.4f}")
print(f"  mAP@50-95 : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")

## 6. 可视化预测结果

In [ ]:
import glob
from IPython.display import Image, display

# 对测试集前 6 张图进行预测
test_images = sorted(glob.glob(os.path.join(DATASET_ROOT, 'test', 'images', '*.jpg')))[:6]

pred_results = model_best.predict(
    source=test_images,
    conf=0.25,
    save=True,
    project='/kaggle/working/runs',
    name='predict',
)

# 显示预测图
pred_imgs = sorted(glob.glob('/kaggle/working/runs/predict/*.jpg'))
for p in pred_imgs:
    display(Image(filename=p, width=640))

## 7. 打包下载结果

运行后在 Kaggle 右侧 Output 面板下载 `results.zip`

In [ ]:
import shutil

# 复制最优权重到输出根目录方便下载
shutil.copy(best_weights, '/kaggle/working/best.pt')

# 打包训练结果（曲线图、权重等）
shutil.make_archive('/kaggle/working/results', 'zip', results.save_dir)

print('可下载文件：')
for f in os.listdir('/kaggle/working'):
    size = os.path.getsize(f'/kaggle/working/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')